In [ ]:
from darts import TimeSeries

# Target Series with Static Covariates
# Note: In Darts, static covariates are attached to the Target series
all_targets = TimeSeries.from_group_dataframe(
    pandas_df,
    group_cols="PARENT_DEALER_CODE_MODEL_FAMILY",
    value_cols="NET_SALES",
    static_cols=static_cols, # Use the list you defined
    freq='MS'
)

# Future Covariates (only if you have external variables like Price or Holidays)
# If your 'future_covariates' list is empty, Darts' 'add_encoders' will handle 
# the month/year/relative position automatically!
all_future_covs = TimeSeries.from_group_dataframe(
    pandas_df,
    group_cols="PARENT_DEALER_CODE_MODEL_FAMILY",
    value_cols=future_covariates, 
    freq='MS'
) if future_covariates else None

In [ ]:
from darts.dataprocessing.transformers import Scaler
import pandas as pd

# Define cutoff
train_cutoff = pd.Timestamp('2025-10-01')

# 1. Split
train_targets = [s.split_before(train_cutoff)[0] for s in all_targets]
val_targets = [s.split_before(train_cutoff)[1] for s in all_targets]

# 2. Scale Targets (Global Scaling)
# This scales the sales values between 0 and 1
target_scaler = Scaler()
train_targets_transformed = target_scaler.fit_transform(train_targets)
val_targets_transformed = target_scaler.transform(val_targets)

# 3. Scaling Static Covariates (Numeric only)
# Since you label encoded, these are now numbers. 
# While TFT uses embeddings for categories, scaling helps the optimizer.
from darts.dataprocessing.transformers import StaticCovariatesTransformer
static_scaler = StaticCovariatesTransformer()
train_targets_transformed = static_scaler.fit_transform(train_targets_transformed)
val_targets_transformed = static_scaler.transform(val_targets_transformed)

In [ ]:
from darts.models import TFTModel

# Calculate the cardinality (number of unique values) for each categorical column
# The model needs this to size the "embedding" brain correctly
categorical_embedding_sizes = {
    col: len(encoders[col].classes_) for col in static_cols
}

model = TFTModel(
    input_chunk_length=12,
    output_chunk_length=6,
    hidden_size=64,
    lstm_layers=2,
    num_attention_heads=4,
    dropout=0.1,
    batch_size=256,
    n_epochs=50,
    
    # CRITICAL: Tell TFT which static cols are categorical
    categorical_static_covariates=static_cols,
    
    # Pass your pre-defined encoders (cyclic month, etc.)
    add_encoders=add_encoders,
    
    # PyTorch Lightning kwargs for performance
    pl_trainer_kwargs={"accelerator": "gpu", "devices": [0]} # Use GPU if available!
)

In [ ]:
model.fit(
    series=train_targets_transformed,
    val_series=val_targets_transformed,
    future_covariates=all_future_covs_transformed if all_future_covs else None,
    verbose=True
)

### Claude optimized code

In [ ]:
from darts import TimeSeries
from darts.dataprocessing.transformers import Scaler, StaticCovariatesTransformer
from darts.models import TFTModel
import pandas as pd

# ==================================================
# 1. Build TimeSeries (parallelized)
# ==================================================
all_targets = TimeSeries.from_group_dataframe(
    pandas_df,
    group_cols="PARENT_DEALER_CODE_MODEL_FAMILY",
    value_cols="NET_SALES",
    static_cols=static_cols,
    freq='MS',
    n_jobs=-1  # parallel
)

all_future_covs = TimeSeries.from_group_dataframe(
    pandas_df,
    group_cols="PARENT_DEALER_CODE_MODEL_FAMILY",
    value_cols=future_covariates,
    freq='MS',
    n_jobs=-1
) if future_covariates else None

# ==================================================
# 2. Split (train/val cutoff)
# ==================================================
train_cutoff = pd.Timestamp('2025-10-01')

# Split into train and test
train_targets = [s.split_before(train_cutoff)[0] for s in all_targets]
test_targets  = [s.split_before(train_cutoff)[1] for s in all_targets]

# Future covariates — keep full series (model needs future values at predict time)
if all_future_covs:
    train_future_covs = [s.split_before(train_cutoff)[0] for s in all_future_covs]
    full_future_covs  = all_future_covs  # used at predict time
else:
    train_future_covs = None
    full_future_covs = None

# ==================================================
# 3. Scale (parallelized)
# ==================================================
target_scaler = Scaler(n_jobs=-1)
train_targets_scaled = target_scaler.fit_transform(train_targets)
val_targets_scaled   = target_scaler.transform(val_targets)

static_scaler = StaticCovariatesTransformer()
train_targets_scaled = static_scaler.fit_transform(train_targets_scaled)
val_targets_scaled   = static_scaler.transform(val_targets_scaled)

# Scale future covariates too (important!)
if all_future_covs:
    future_cov_scaler = Scaler(n_jobs=-1)
    train_future_covs_scaled = future_cov_scaler.fit_transform(train_future_covs)
    val_future_covs_scaled   = future_cov_scaler.transform(val_future_covs)
else:
    train_future_covs_scaled = None
    val_future_covs_scaled = None

# ==================================================
# 4. Model (lean config for fast iteration)
# ==================================================
categorical_embedding_sizes = {
    col: len(encoders[col].classes_) for col in static_cols
}

model = TFTModel(
    input_chunk_length=12,
    output_chunk_length=6,
    hidden_size=32,          # reduced from 64 for speed
    lstm_layers=1,           # reduced from 2
    num_attention_heads=2,   # reduced from 4
    dropout=0.1,
    batch_size=512,          # increased — GPU can handle it
    n_epochs=10,             # reduced — just to prove it works
    categorical_embedding_sizes=categorical_embedding_sizes,  # bug fix: was passing wrong arg
    add_encoders=add_encoders,
    pl_trainer_kwargs={
        "accelerator": "gpu",
        "devices": [0],
        "precision": "16-mixed",    # mixed precision → much faster on GPU
        "enable_progress_bar": True,
    },
    random_state=42,
)

# ==================================================
# 5. Fit
# ==================================================
model.fit(
    series=train_targets_scaled,
    val_series=val_targets_scaled,
    future_covariates=train_future_covs_scaled,
    val_future_covariates=val_future_covs_scaled,
    verbose=True,
)